# **Question 3: Multi-Class Segmentation for Underwater Imagery - SOLUTION**

In this question, you should finetune a **pretrained U-Net** model for **multi-class segmentation** of underwater images. Your segmentation model will classify each pixel into **one of 8 classes**.

---

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage-3-2026")

print("Path to dataset files:", path)

## **TASK 1: Dataset Class**
- Build a custom dataset class to load images and masks.
- Use Dataloaders to prepare your data.
- Display some images and their corresponding masks.



In [ ]:
import torch

def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)
    
    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val
        
    return remapped_mask

In [ ]:
import os
import glob
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np

class SegDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None, target_transform=None):
        self.image_paths = glob.glob(os.path.join(image_dir, "*.jpg"))
        self.mask_paths = glob.glob(os.path.join(mask_dir, "*.png"))

        self.image_paths.sort()  
        self.mask_paths.sort()

        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        mask = remap_mask(mask)

        return image, mask

In [ ]:
from torch.utils.data import DataLoader
from torch import nn

# Define transforms for images and masks
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])

In [ ]:
# Split the dataset (80% train, 20% test)
import random

image_dir = os.path.join(path, "dataset", "images")
mask_dir = os.path.join(path, "dataset", "masks")

# Get all image paths and shuffle
all_images = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))
all_masks = sorted(glob.glob(os.path.join(mask_dir, "*.png")))

# Create train/test split directories
split_idx = int(0.8 * len(all_images))

# For simplicity, we'll use the same directory but split in the dataset
train_dataset = SegDataset(image_dir, mask_dir, transform=image_transforms, target_transform=mask_transforms)

# Split using torch
train_size = int(0.8 * len(train_dataset))
test_size = len(train_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(train_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

print(f"Training Samples: {len(train_dataset)}, Testing Samples: {len(test_dataset)}")

### Let's display some images

In [ ]:
import matplotlib.pyplot as plt

def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img, 0, 1)
    return img

# Display some images with their masks
for i in range(3):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(denormalize(img))  
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.permute(1, 2, 0), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()

## **TASK 2: Model Class**
- **Use a pretrained U-Net** (from `segmentation_models_pytorch`) with "efficientnet-b0" as an encoder.

In [ ]:
import segmentation_models_pytorch as smp

num_classes = 8  # 8 underwater classes

model = smp.Unet(
    encoder_name="efficientnet-b0",
    encoder_weights="imagenet",
    in_channels=3,
    classes=num_classes
)

print("Model created successfully!")

## **TASK 3: Training and Validation Loops**
- Define the training and validation loops.

In [ ]:
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for images, masks in tqdm(dataloader):
        images = images.to(device)
        masks = masks.squeeze(1).to(device)  # Remove channel dim for CE loss
        
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.squeeze(1).to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

## **TASK 4: Running Training**
- Define the loss and the optimizer.
- Train the model for 10 epochs.
- Print the training and validation losses.
- Plot loss curve.

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")

In [ ]:
# Plot Loss
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

## **TASK 5: Visualizing Predictions**
- Visualize your model's predictions against the ground truth for several images.

In [ ]:
model.eval()

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

with torch.no_grad():
    for i in range(3):
        image, mask = test_dataset[i]
        
        # Get prediction
        pred = model(image.unsqueeze(0).to(device))
        pred = pred.argmax(dim=1).squeeze().cpu().numpy()
        
        # Denormalize image
        img = denormalize(image)
        
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Image')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(mask.squeeze().numpy(), cmap='tab10')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(pred, cmap='tab10')
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')

plt.tight_layout()
plt.show()